In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FileUpload
from PIL import Image
import io


In [ ]:
uploader = FileUpload(accept='image/*', multiple=False)
display(uploader)

def get_image():
    if len(uploader.value) == 0:
        return None
    file = next(iter(uploader.value.values()))
    img = Image.open(io.BytesIO(file['content'])).convert("L")
    return np.array(img)


In [ ]:
img = get_image()
if img is not None:
    plt.figure(figsize=(4,4))
    plt.imshow(img, cmap='gray')
    plt.axis("off")
    plt.title("Uploaded Image")
else:
    print("Upload an image to continue.")


In [ ]:
if img is not None:
    f = np.fft.fft2(img)
    fshift = np.fft.fftshift(f)
    magnitude = np.log(1 + np.abs(fshift))

    plt.figure(figsize=(5,5))
    plt.imshow(magnitude, cmap='gray')
    plt.axis("off")
    plt.title("FFT Magnitude Spectrum")


In [ ]:
def filter_fft(crop_radius=50):
    if img is None:
        print("No image uploaded.")
        return
    rows, cols = img.shape
    crow, ccol = rows//2, cols//2
    mask = np.zeros_like(img)
    mask[crow-crop_radius:crow+crop_radius, ccol-crop_radius:ccol+crop_radius] = 1
    filtered = fshift * mask
    recon = np.fft.ifft2(np.fft.ifftshift(filtered))
    recon = np.abs(recon)
    fig, ax = plt.subplots(1,2, figsize=(8,4))
    ax[0].imshow(mask, cmap='gray'); ax[0].set_title("Frequency Mask"); ax[0].axis("off")
    ax[1].imshow(recon, cmap='gray'); ax[1].set_title("Reconstructed Image"); ax[1].axis("off")
    plt.show()

interact(filter_fft, crop_radius=IntSlider(min=5, max=200, step=5, value=40))
